# 05 — Random Forest: Modelo Candidato à Inferência

Este notebook treina e avalia um **Random Forest** como o segundo modelo candidato
para o endpoint de inferência do projeto.

**Por que Random Forest?**
- Captura interações não-lineares entre features sem precisar de arquitetura manual
- Robusto a outliers e não exige StandardScaler
- Feature importance nativa — interpretabilidade para o negócio
- Forte baseline para datasets tabulares pequenos/médios (~7k linhas)

**Configuração canônica** (mesma da Fase 3 — ADR-009 e ADR-010):
- Split: **80/10/10** (`test_size=0.10`, `val_size=0.10`)
- Feature engineering: **`ohe`** (35 features)
- Busca de hiperparâmetros: `RandomizedSearchCV` (n_iter=20, 5-fold)
- Run MLflow: `rf_8010_ohe`

**Meta:** blind test AUC > 0.8651 (melhor MLP da bateria — `mlp_8010_ohe_b16`).

**Pré-requisito:** `notebooks/03_baseline.ipynb` executado (baselines no MLflow).

**Seções:**
1. Carregamento dos Dados
2. Configuração do MLflow
3. Busca de Hiperparâmetros — RandomizedSearchCV
4. Treino Final e Avaliação no Blind Test
5. Feature Importance
6. Comparação Final — Todos os Modelos
7. Análise de Custo — Threshold Ótimo
8. Conclusão — Escolha do Modelo para a API

In [ ]:
from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

from churn.config import (
    AUTHOR,
    COST_FALSE_NEGATIVE,
    COST_FALSE_POSITIVE,
    DATASET_VERSION,
    MLFLOW_EXPERIMENT_NAME,
    ROC_AUC_TARGET,
    ROOT_DIR,
    SEED,
)
from churn.data.loader import load_raw_data
from churn.data.preprocessing import (
    build_preprocessing_pipeline,
    clean_raw,
    split_features_target,
    stratified_split,
)
from churn.training.tracking import setup_mlflow

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Carregamento dos Dados

Mesma configuração canônica da Fase 3 (ADR-009 + ADR-010):
- Split **80/10/10** estratificado (seed=42)
- Feature engineering **`ohe`** — 35 features pós-pipeline

In [ ]:
df_raw = load_raw_data()
df_clean = clean_raw(df_raw)
X, y = split_features_target(df_clean)

splits = stratified_split(X, y, test_size=0.10, val_size=0.10)

print(f"Treino (80%): {splits.X_train.shape[0]:>5,} linhas | churn = {splits.y_train.mean():.1%}")
print(f"Val    (10%): {splits.X_val.shape[0]:>5,} linhas | churn = {splits.y_val.mean():.1%}")
print(f"Teste  (10%): {splits.X_test.shape[0]:>5,} linhas | churn = {splits.y_test.mean():.1%}")

# Fit preprocessor on train and transform all splits
preprocessor = build_preprocessing_pipeline(tenure_variant="ohe")
X_train = preprocessor.fit_transform(splits.X_train)
X_val = preprocessor.transform(splits.X_val)
X_test = preprocessor.transform(splits.X_test)
y_train = splits.y_train.to_numpy()
y_val = splits.y_val.to_numpy()
y_test = splits.y_test.to_numpy()

print(f"\nFeatures pós-pipeline (ohe): {X_train.shape[1]}")

## 2. Configuração do MLflow

In [ ]:
setup_mlflow(tracking_uri=(ROOT_DIR / "mlruns").as_uri())
print(f"Experimento : {MLFLOW_EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

RF_RUN_NAME = "rf_8010_ohe"

## 3. Busca de Hiperparâmetros — RandomizedSearchCV

`RandomizedSearchCV` com `n_iter=20` e `cv=5` estratificados busca eficientemente
no espaço de hiperparâmetros sem precisar de 50+ runs manuais.

**Espaço de busca:**
| Parâmetro | Range | Nota |
|-----------|-------|------|
| `n_estimators` | [100, 200, 300, 500] | Mais árvores = mais estável |
| `max_depth` | [None, 10, 20, 30] | None = árvores completas |
| `min_samples_split` | [2, 5, 10] | Regularização por profundidade |
| `min_samples_leaf` | [1, 2, 4] | Regularização por folhas |
| `max_features` | ['sqrt', 'log2', 0.5] | Feature subsampling |
| `class_weight` | `'balanced'` | Fixo — compensa 26%/74% imbalance |

In [ ]:
# Check for existing run
existing_rf = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    filter_string=f"tags.mlflow.runName = '{RF_RUN_NAME}'",
    order_by=["start_time DESC"],
)

if len(existing_rf) > 0:
    rf_run_id = existing_rf.iloc[0]["run_id"]
    print(f"Run existente encontrado: {rf_run_id}")
    rf_data = mlflow.get_run(rf_run_id).data
    best_params = {k.replace("param_", ""): v for k, v in rf_data.params.items()
                   if k.startswith("best_")}
    print(f"\nMétricas do run existente:")
    for k, v in rf_data.metrics.items():
        try:
            print(f"  {k}: {float(v):.4f}")
        except (ValueError, TypeError):
            pass
else:
    print("Iniciando RandomizedSearchCV...")
    param_grid = {
        "n_estimators": [100, 200, 300, 500],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.5],
    }

    base_rf = RandomForestClassifier(
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEED,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    search = RandomizedSearchCV(
        base_rf,
        param_distributions=param_grid,
        n_iter=20,
        cv=cv,
        scoring="roc_auc",
        refit=True,
        n_jobs=-1,
        random_state=SEED,
        verbose=1,
    )
    search.fit(X_train, y_train)

    best_params = search.best_params_
    print(f"\nMelhores hiperparâmetros encontrados:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    print(f"\nCV AUC (search, treino only): {search.best_score_:.4f}")

## 4. Treino Final e Avaliação no Blind Test

Com os melhores hiperparâmetros encontrados, treinamos o RF final em todo o
conjunto de treino (80%) e avaliamos no blind test (10%) — nunca usado antes.

In [ ]:
import tempfile
import joblib
from pathlib import Path

if len(existing_rf) > 0:
    # Load model from existing MLflow run
    rf_final = mlflow.sklearn.load_model(f"runs:/{rf_run_id}/model")
    print(f"Modelo carregado do run existente: {rf_run_id}")
else:
    # Train final model with best params on full train set
    rf_final = RandomForestClassifier(
        **best_params,
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEED,
    )
    rf_final.fit(X_train, y_train)
    print("Modelo final treinado com melhores hiperparâmetros.")

# Evaluate on holdout val
y_val_proba = rf_final.predict_proba(X_val)[:, 1]
y_val_pred = rf_final.predict(X_val)
val_roc_auc = float(roc_auc_score(y_val, y_val_proba))
val_pr_auc = float(average_precision_score(y_val, y_val_proba))
val_f1 = float(f1_score(y_val, y_val_pred, zero_division=0))
val_prec = float(precision_score(y_val, y_val_pred, zero_division=0))
val_recall = float(recall_score(y_val, y_val_pred, zero_division=0))

# Evaluate on blind test
y_test_proba = rf_final.predict_proba(X_test)[:, 1]
y_test_pred = rf_final.predict(X_test)
blind_roc_auc = float(roc_auc_score(y_test, y_test_proba))
blind_pr_auc = float(average_precision_score(y_test, y_test_proba))
blind_f1 = float(f1_score(y_test, y_test_pred, zero_division=0))
blind_prec = float(precision_score(y_test, y_test_pred, zero_division=0))
blind_recall = float(recall_score(y_test, y_test_pred, zero_division=0))

print("\nHoldout Val (10%):")
print(f"  ROC-AUC  : {val_roc_auc:.4f}")
print(f"  PR-AUC   : {val_pr_auc:.4f}")
print(f"  F1       : {val_f1:.4f}")
print(f"  Recall   : {val_recall:.4f}")
print("\nBlind Test (10%) — número desenviesado:")
print(f"  ROC-AUC  : {blind_roc_auc:.4f}")
print(f"  PR-AUC   : {blind_pr_auc:.4f}")
print(f"  F1       : {blind_f1:.4f}")
print(f"  Recall   : {blind_recall:.4f}")
print(f"\nMeta MLP blind AUC: 0.8651  |  RF blind AUC: {blind_roc_auc:.4f}  ",
      "✓ META ATINGIDA" if blind_roc_auc > 0.8651 else "✗ ABAIXO DA META")

In [ ]:
if len(existing_rf) == 0:
    # Log run to MLflow
    with mlflow.start_run(run_name=RF_RUN_NAME) as run:
        rf_run_id = run.info.run_id

        mlflow.log_params({
            "model_type": "RandomForestClassifier",
            "class_weight": "balanced",
            "seed": SEED,
            "dataset_version": DATASET_VERSION,
            "tenure_variant": "ohe",
            "split": "80/10/10",
            "n_features": int(X_train.shape[1]),
            "search_n_iter": 20,
            "cv_folds": 5,
            **{f"best_{k}": str(v) for k, v in best_params.items()},
        })
        mlflow.set_tags({
            "model_type": "RandomForestClassifier",
            "dataset_version": DATASET_VERSION,
            "author": AUTHOR,
        })

        mlflow.log_metrics({
            "holdout_val_roc_auc": val_roc_auc,
            "holdout_val_pr_auc": val_pr_auc,
            "holdout_val_f1": val_f1,
            "holdout_val_precision": val_prec,
            "holdout_val_recall": val_recall,
            "blind_test_roc_auc": blind_roc_auc,
            "blind_test_pr_auc": blind_pr_auc,
            "blind_test_f1": blind_f1,
            "blind_test_precision": blind_prec,
            "blind_test_recall": blind_recall,
        })

        # Confusion matrix artifact
        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_val, y_val_pred, ax=ax, colorbar=False, cmap="Blues"
        )
        ax.set_title("Confusion matrix (val holdout)")
        fig.tight_layout()
        with tempfile.TemporaryDirectory() as tmpdir:
            cm_path = Path(tmpdir) / "confusion_matrix.png"
            fig.savefig(cm_path, dpi=120, bbox_inches="tight")
            mlflow.log_artifact(str(cm_path))
        plt.close(fig)

        # Log model and preprocessor
        mlflow.sklearn.log_model(rf_final, "model")
        with tempfile.TemporaryDirectory() as tmpdir:
            preproc_path = Path(tmpdir) / "preprocessor.joblib"
            joblib.dump(preprocessor, preproc_path)
            mlflow.log_artifact(str(preproc_path))

    print(f"Run logado: {RF_RUN_NAME} ({rf_run_id})")
else:
    print(f"Run existente reutilizado: {rf_run_id}")

## 5. Feature Importance

Random Forest fornece importância de feature nativa via Gini impurity.
As top features confirmam (ou questionam) as hipóteses da EDA.

In [ ]:
# Feature names from the fitted preprocessor
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = [f"feat_{i}" for i in range(X_train.shape[1])]

importances = rf_final.feature_importances_
fi_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top-15 features por importância (Gini):")
display(fi_df.head(15))

# Plot
top_n = 20
top_fi = fi_df.head(top_n)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_fi["feature"][::-1], top_fi["importance"][::-1], color="#1565C0", edgecolor="white")
ax.set_xlabel("Importância (Gini)")
ax.set_title(f"Random Forest — Top {top_n} Features por Importância", fontsize=12)
plt.tight_layout()
plt.savefig("rf_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Comparação Final — Todos os Modelos

Compara o RF com os modelos anteriores: Dummy, melhor LogReg e MLP.
Métrica primária: **blind test ROC-AUC** (quando disponível) ou holdout.

In [ ]:
COMPARISON_RUNS = [
    "dummy_baseline",
    "logreg_baseline",
    "logreg_nophone_noml_8010_le",
    "mlp_8010_ohe_b16",
    RF_RUN_NAME,
]

runs_all = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    order_by=["start_time DESC"],
)

col_map = {
    "tags.mlflow.runName": "modelo",
    "metrics.roc_auc_mean": "roc_auc_cv",
    "metrics.holdout_val_roc_auc": "roc_auc_holdout",
    "metrics.blind_test_roc_auc": "roc_auc_blind",
    "metrics.pr_auc_mean": "pr_auc_cv",
    "metrics.holdout_val_pr_auc": "pr_auc_holdout",
    "metrics.f1_mean": "f1_cv",
    "metrics.holdout_val_f1": "f1_holdout",
}

available = [c for c in col_map if c in runs_all.columns]
mask = runs_all["tags.mlflow.runName"].isin(COMPARISON_RUNS)

all_comparison = (
    runs_all[mask][available]
    .rename(columns={k: col_map[k] for k in available})
    .drop_duplicates(subset="modelo", keep="first")
    .sort_values("roc_auc_holdout", ascending=False)
    .reset_index(drop=True)
)

with pd.option_context("display.float_format", "{:.4f}".format, "display.max_columns", 20):
    display(all_comparison)

In [ ]:
# Bar chart: holdout AUC comparison (all models)
plot_df = all_comparison.dropna(subset=["roc_auc_holdout"]).sort_values(
    "roc_auc_holdout", ascending=True
)

palette = [
    "#B71C1C" if "rf" in str(m) else
    "#1B5E20" if "mlp" in str(m) else
    "#2196F3" if "logreg" in str(m) else
    "#9E9E9E"
    for m in plot_df["modelo"]
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Holdout AUC
bars = axes[0].barh(plot_df["modelo"], plot_df["roc_auc_holdout"], color=palette, edgecolor="white", height=0.55)
axes[0].axvline(ROC_AUC_TARGET, color="red", linestyle="--", linewidth=1.5, label=f"Alvo ({ROC_AUC_TARGET:.2f})")
axes[0].bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
axes[0].set_title("ROC-AUC — Holdout", fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8)

# Blind test AUC (where available)
blind_df = all_comparison.dropna(subset=["roc_auc_blind"]).sort_values("roc_auc_blind", ascending=True)
if len(blind_df) > 0:
    palette_blind = [
        "#B71C1C" if "rf" in str(m) else
        "#1B5E20" if "mlp" in str(m) else
        "#2196F3" if "logreg" in str(m) else
        "#9E9E9E"
        for m in blind_df["modelo"]
    ]
    bars2 = axes[1].barh(blind_df["modelo"], blind_df["roc_auc_blind"], color=palette_blind, edgecolor="white", height=0.55)
    axes[1].bar_label(bars2, fmt="%.4f", padding=4, fontsize=9)
    axes[1].set_title("ROC-AUC — Blind Test", fontsize=11, fontweight="bold")
else:
    axes[1].set_visible(False)

plt.suptitle("Comparação Final — Todos os Modelos (ohe, 80/10/10)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("final_model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Análise de Custo — Threshold Ótimo

Mesma análise de custo do notebook 04, aplicada ao RF.

| Erro | Custo |
|------|-------|
| **FP** — ação de retenção desnecessária | R$ 50 |
| **FN** — cliente perdido sem intervenção | R$ 500 |

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
costs, fp_counts, fn_counts = [], [], []

for thresh in thresholds:
    y_pred_t = (y_val_proba >= thresh).astype(int)
    fp = int(((y_pred_t == 1) & (y_val == 0)).sum())
    fn = int(((y_pred_t == 0) & (y_val == 1)).sum())
    costs.append(fp * COST_FALSE_POSITIVE + fn * COST_FALSE_NEGATIVE)
    fp_counts.append(fp)
    fn_counts.append(fn)

costs = np.array(costs)
opt_idx = int(np.argmin(costs))
rf_optimal_threshold = float(thresholds[opt_idx])
rf_optimal_cost = float(costs[opt_idx])

ref_idx = int(np.argmin(np.abs(thresholds - 0.5)))
rf_ref_cost = float(costs[ref_idx])
rf_cost_saving = rf_ref_cost - rf_optimal_cost

print(f"Threshold padrão  (0.50): custo = R$ {rf_ref_cost:>10,.0f}")
print(f"Threshold ótimo  ({rf_optimal_threshold:.2f}): custo = R$ {rf_optimal_cost:>10,.0f}")
print(f"Economia estimada       : R$ {rf_cost_saving:>10,.0f}  ({rf_cost_saving / rf_ref_cost:.1%})")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, costs, color="#B71C1C", linewidth=2, label="Custo total (RF)")
ax.fill_between(thresholds, costs, costs.min(), alpha=0.08, color="#B71C1C")
ax.axvline(rf_optimal_threshold, color="#E53935", linestyle="--", linewidth=2,
           label=f"Ótimo = {rf_optimal_threshold:.2f}  (R$ {rf_optimal_cost:,.0f})")
ax.axvline(0.5, color="gray", linestyle=":", linewidth=1.5,
           label=f"Padrão = 0.50  (R$ {rf_ref_cost:,.0f})")
ax.set_xlabel("Threshold")
ax.set_ylabel("Custo total estimado (R$)")
ax.set_title(f"Análise de Custo — RF\nFP=R${COST_FALSE_POSITIVE:.0f} | FN=R${COST_FALSE_NEGATIVE:.0f}",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(lambda x, _: f"R${x:,.0f}")
plt.tight_layout()
plt.show()

## 8. Conclusão — Escolha do Modelo para a API

In [ ]:
MLP_BLIND_REF = 0.8651  # mlp_8010_ohe_b16 blind test (bateria)
LOGREG_HOLDOUT_REF = 0.8725  # logreg_nophone_noml_8010_le holdout (bateria)

sep = "=" * 64
print(sep)
print("  CONCLUSÃO — ESCOLHA DO MODELO PARA A API")
print(sep)
print()
print("  Comparativo de blind test (estimativa desenviesada):")
print(f"    MLP  (mlp_8010_ohe_b16)          : {MLP_BLIND_REF:.4f}")
print(f"    RF   ({RF_RUN_NAME})            : {blind_roc_auc:.4f}")
print(f"    Delta RF − MLP                    : {blind_roc_auc - MLP_BLIND_REF:+.4f}")
print()

if blind_roc_auc > MLP_BLIND_REF:
    winner = "Random Forest"
    winner_run = RF_RUN_NAME
    print(f"  Vencedor: {winner}")
    print(f"  O RF supera o MLP em blind test — usar RF no endpoint da API.")
elif blind_roc_auc > MLP_BLIND_REF - 0.002:
    winner = "Empate técnico"
    winner_run = RF_RUN_NAME
    print(f"  Resultado: {winner} (diferença < 0.002)")
    print(f"  RF escolhido por ter feature importance nativa — maior interpretabilidade.")
else:
    winner = "MLP"
    winner_run = "mlp_8010_ohe_b16"
    print(f"  Vencedor: {winner}")
    print(f"  O MLP supera o RF em blind test — usar MLP no endpoint da API.")

print()
print(f"  Modelo escolhido para Fase 4 (FastAPI): {winner_run}")
print(f"  Threshold ótimo para deploy: {rf_optimal_threshold:.2f} (RF)")
print()
print(sep)
print("  Próximo: notebook 06 — FastAPI /predict endpoint (Fase 4)")
print(sep)